# ㄴ/ㄹ 삽입 **환경** 검색 v2 (로마자 기반)

**작성일**: 2026-02-10  
**목적**: 우리말샘 사전 형태소 경계 + 로마자 표기 활용  
**관련 문서**: `REVIEW_GUIDE_V2.md`, `34_n_insertion_v2.md`

---

## 🎯 v1 → v2 개선 사항

| 항목 | v1 | v2 |
|------|-----|-----|
| 형태소 출처 | seg_morph (자동 분석) | **dict_morph** (우리말샘 원본) ⭐ |
| 자음/모음 판별 | 한글 유니코드 | **word_roman** (로마자) ⭐ |
| ㄹ 삽입 | 없음 | **포함** (물엿[물렫]) ⭐ |
| 비교 정보 | seg_morph만 | **anal_morph 추가** ⭐ |
| 자동 감지 | 없음 | **pron_roman 기반** ⭐ |
| 결과 | "경+우", "필+요" 포함 | 사전 복합명사만 |

---

## ⚠️ 중요: 이 검색의 목적

**"ㄴ/ㄹ 삽입 환경"을 찾는 것**

### 검색 조건 (환경)
- **자음 종성 + i/j(y) 초성**
- 로마자: 자음 ending + **i, y 시작**
- **w는 제외!** (와, 워, 위 등은 ㄴ 삽입 환경 아님)
- **a, e, o, u 제외!** (아, 어, 오, 우 등은 ㄴ 삽입 환경 아님)

### 포함되는 케이스

#### ✅ 삽입이 일어나는 경우
```
솜이불 → [솜니불]  (ㄴ 삽입 O)
물엿 → [물렫]      (ㄹ 삽입 O)
```

#### ✅ 삽입이 안 일어나는 경우 (환경만 맞음)
```
밤일 → 서울: [밤닐] (ㄴ 삽입 O) vs 경상: [바밀] (ㅁ 약화)
```

**→ 모두 중요한 데이터!** `n_insertion` 컬럼에 yes/no/dialect 등을 채워야 함

---

## 🔧 주요 기술적 개선

### 1. 음절 vs 형태소 문제 해결 ⭐

**문제**: word_roman은 음절 단위, dict_morph는 형태소 단위
```
솜이불:
  - dict_morph: "솜-이불" (2개 형태소)
  - word_roman: "SOm-I-BUl" (3개 음절)
  - 기존 코드: 2 ≠ 3 → 제외됨 ❌
```

**해결**: `parse_word_roman_by_morphemes()` 함수
```
음절을 형태소 글자 수에 따라 재구성:
  ["SOm", "I-BUl"] (형태소별로 묶음)
  → 이제 2 == 2 ✅
```

### 2. i/j(y) 계열만 정확히 검색 ⭐

```python
def starts_with_i_j_roman(roman_str):
    """i, j(y)로 시작하는지 - w 제외!"""
    return first_char in ['i', 'y']
```

**포함**: 이(I), 야/여/요/유(iA/yA, iEO/yEO, iO/yO, iU/yU)  
**제외**: 와/워/위(wA, wEO, wI), 아/어/오/우(A, EO, O, U)

### 3. pron_roman 기반 자동 감지 ⭐

```
word_roman: "SOm-I-BUl"
pron_roman: "SOm-NI-BUl"
→ I → NI (N 추가) → detected_insertion: 'n_inserted'
```

**자동 감지 값**:
- `n_inserted`: ㄴ 삽입 확인
- `l_inserted`: ㄹ 삽입 확인
- `no`: 삽입 안 됨 (경음화 등)
- `unknown`: 발음 정보 없음 (수동 검토 필요)

---

## 📊 출력 정보

### 기본 정보
- word, word_stem, sense_id, definition

### 발음 정보 ⭐
- pron, pron_roman
- **detected_insertion** (자동 감지)
- insertion_type (ㄴ삽입/ㄹ삽입)
- n_in_pron, l_in_pron (참고용)

### 형태소 분석 (비교용)
- **dict_morph** (우리말샘 원본) ← 메인
- word_roman (로마자)
- morph1, morph2, morph1_roman, morph2_roman
- seg_morph, anal_morph (참조)

### 빈도 정보
- LS: messenger, written, spoken, total
- MP: written, spoken, total
- 형태소 빈도: freq_morph1_LS, freq_morph2_LS

### 검토용 컬럼
- review_status, morph_ok
- **n_insertion** ← 수동으로 채워야 함!
- decision, notes

---

## 🚀 다음 단계

1. **이 노트북 실행** → CSV 생성
2. **Google Sheets 열기** → 드롭다운 설정
3. **수동 검토** → REVIEW_GUIDE_V2.md 참고
4. **필터링** → decision = "keep"만 추출
5. **Seoul Corpus 확인** → 실제 발음 검증

---

## 📚 관련 문서

- **REVIEW_GUIDE_V2.md**: 수동 검토 가이드 (필수!) ⭐
- **34_n_insertion_v2.md**: 전체 개요
- **FINAL_CORRECTION_I_J_ONLY.md**: 최종 조건 정리
- **BUG_FIX_V2_SYLLABLE_VS_MORPHEME.md**: 버그 수정 내역

---

**핵심**: 환경 조건(자음 + i/j)을 만족하지만 실제 삽입이 안 되는 경우도 중요한 데이터입니다!

## 1️⃣ 환경 설정

In [44]:
# 1.1 Google Drive 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("로컬 환경")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
# 1.2 경로 설정
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
else:
    PROJECT_ROOT = 'g:/내 드라이브/DATA_2026'

V7_LEXICON = f'{PROJECT_ROOT}/10_dictionary_build/output/04_v7_lexicon.csv'
MORPHEME_FREQ = f'{PROJECT_ROOT}/00_raw_data/02_nikl_ls/08_ALL_morpheme_freq.csv'
RESULT_DIR = f'{PROJECT_ROOT}/30_search_dictionary/search_results'

print(f"v7: {V7_LEXICON}")
print(f"형태소 빈도: {MORPHEME_FREQ}")
print(f"결과: {RESULT_DIR}")

v7: /content/drive/MyDrive/DATA_2026/10_dictionary_build/output/04_v7_lexicon.csv
형태소 빈도: /content/drive/MyDrive/DATA_2026/00_raw_data/02_nikl_ls/08_ALL_morpheme_freq.csv
결과: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results


In [46]:
# 1.3 라이브러리
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import re
import os

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

# 공유 유틸리티 모듈 로드
os.chdir(f'{PROJECT_ROOT}/30_search_dictionary')
%run utils_phonology.py
print("준비 완료")

[OK] utils_phonology.py 로드 완료
   함수 34개
준비 완료


---

## 2️⃣ 데이터 로드

In [47]:
# 2.1 v7 Lexicon 로드
df_v7 = pd.read_csv(V7_LEXICON, encoding='utf-8-sig', low_memory=False)
print(f"v7 Lexicon: {len(df_v7):,}개")
print(f"dict_morph 있는 행: {df_v7['dict_morph'].notna().sum():,}개")

v7 Lexicon: 528,088개
dict_morph 있는 행: 323,146개


In [48]:
# 2.2 형태소 빈도 로드
df_morph_freq = pd.read_csv(MORPHEME_FREQ, encoding='utf-8-sig')
print(f"\n형태소 빈도 데이터: {len(df_morph_freq):,}개")

# 형태소별 총 빈도 계산
morph_freq_agg = df_morph_freq.groupby('word')['freq'].sum().to_dict()
print(f"고유 형태소: {len(morph_freq_agg):,}개")

/tmp/ipykernel_195/3269206556.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_morph_freq = pd.read_csv(MORPHEME_FREQ, encoding='utf-8-sig')



형태소 빈도 데이터: 146,028개
고유 형태소: 91,084개


# 2.3 형태소 어종 딕셔너리 구축 ⭐

**중요**: 각 형태소의 어종을 정확하게 파악하기 위해 v7에서 단일 형태소 추출

In [49]:
# 어종 딕셔너리 구축 (sense_no 기반 - 동음이의어 문제 해소)
sense_origin_dict = build_sense_origin_dict(df_v7)

sense_no 어종 딕셔너리: 74개


---

## 3️⃣ 로마자 기반 분석 함수

In [50]:
# 로마자 분석 함수: utils_phonology.py에서 로드됨
# ends_with_consonant_roman, starts_with_i_j_roman, etc.
print("로마자 분석 함수: utils_phonology.py에서 로드 완료")

로마자 분석 함수: utils_phonology.py에서 로드 완료


In [51]:
# 형태소 파싱 함수: utils_phonology.py에서 로드됨
# parse_dict_morph, parse_dict_morph_with_boundaries, parse_word_roman_by_morphemes
print("형태소 파싱 함수: utils_phonology.py에서 로드 완료")

형태소 파싱 함수: utils_phonology.py에서 로드 완료


---

## 4️⃣ ㄴ/ㄹ 삽입 환경 검색

In [52]:
# 검색/분류 함수: utils_phonology.py에서 로드됨
# check_n_l_insertion_env, detect_insertion_from_pron
# classify_compound_type, get_compound_type_description
# NEW: check_n_l_insertion_internal (단어내부 ㄴ삽입)
print("검색/분류 함수: utils_phonology.py에서 로드 완료")

검색/분류 함수: utils_phonology.py에서 로드 완료


In [53]:
# 3.3 합성어 유형 분류 - 모든 어종 조합 고려
def classify_compound_type(morph1_origin, morph2_origin):
    """
    합성어 유형 분류 - 모든 어종 조합을 세분화

    어종 약어:
    - N: 고유어 (Native)
    - S: 한자어 (Sino-Korean)
    - L: 외래어 (Loanword)
    - H: 혼종어 (Hybrid)
    - U: 불명 (Unknown)

    Returns:
        약어 조합 (예: "N+N", "S+N", "N+S")
    """
    # 어종 → 약어 매핑
    origin_map = {
        '고유어': 'N',
        '한자어': 'S',
        '외래어': 'L',
        '혼종어': 'H',
        'unknown': 'U'
    }

    abbr1 = origin_map.get(morph1_origin, 'U')
    abbr2 = origin_map.get(morph2_origin, 'U')

    return f"{abbr1}+{abbr2}"

def get_compound_type_description(compound_type):
    """
    약어 조합 → 한글 설명

    주요 유형:
    - N+N: 고유어+고유어 (ㄴ삽입/경음화 활발)
    - S+S: 한자어+한자어 (유음화 활발, ㄴ삽입/경음화 제한적)
    - S+N: 한자어+고유어 (경음화 활발)
    - N+S: 고유어+한자어 (제한적)
    - L+N: 외래어+고유어 (경음화)
    """
    desc_map = {
        'N+N': '고유어+고유어',
        'S+S': '한자어+한자어',
        'N+S': '고유어+한자어',
        'S+N': '한자어+고유어',
        'L+N': '외래어+고유어',
        'N+L': '고유어+외래어',
        'L+L': '외래어+외래어',
        'S+L': '한자어+외래어',
        'L+S': '외래어+한자어',
        'H+N': '혼종어+고유어',
        'N+H': '고유어+혼종어',
        'H+S': '혼종어+한자어',
        'S+H': '한자어+혼종어',
        'H+L': '혼종어+외래어',
        'L+H': '외래어+혼종어',
        'H+H': '혼종어+혼종어',
    }
    return desc_map.get(compound_type, compound_type)

# 테스트
print("합성어 유형 분류 테스트 (세분화):")
test_cases = [
    ('고유어', '고유어', 'N+N'),
    ('한자어', '한자어', 'S+S'),
    ('고유어', '한자어', 'N+S'),
    ('한자어', '고유어', 'S+N'),
    ('외래어', '고유어', 'L+N'),
    ('고유어', '외래어', 'N+L'),
    ('외래어', '외래어', 'L+L'),
    ('한자어', '외래어', 'S+L'),
    ('혼종어', '고유어', 'H+N'),
    ('unknown', '고유어', 'U+N'),
]

for m1, m2, expected in test_cases:
    result = classify_compound_type(m1, m2)
    desc = get_compound_type_description(result)
    status = "✓" if result == expected else "✗"
    print(f"  {status} {m1}+{m2}: {result} ({desc})")

합성어 유형 분류 테스트 (세분화):
  ✓ 고유어+고유어: N+N (고유어+고유어)
  ✓ 한자어+한자어: S+S (한자어+한자어)
  ✓ 고유어+한자어: N+S (고유어+한자어)
  ✓ 한자어+고유어: S+N (한자어+고유어)
  ✓ 외래어+고유어: L+N (외래어+고유어)
  ✓ 고유어+외래어: N+L (고유어+외래어)
  ✓ 외래어+외래어: L+L (외래어+외래어)
  ✓ 한자어+외래어: S+L (한자어+외래어)
  ✓ 혼종어+고유어: H+N (혼종어+고유어)
  ✓ unknown+고유어: U+N (U+N)


In [ ]:
# 4.2 검색 함수
def search_n_l_insertion_candidates(df, sense_origin_dict):
    """
    v7에서 ㄴ/ㄹ 삽입 환경 추출 + 사전 발음 기반 자동 판단 + 경계 유형 정보
    + 조건 변수 컬럼 추가 + 단어 내부 ㄴ삽입 검색

    detected_insertion 값:
    - n_inserted: 사전 발음에서 ㄴ 삽입 확인
    - l_inserted: 사전 발음에서 ㄹ 삽입 확인
    - no: pron 있지만 삽입 아닌 다른 현상
    - no_pron_same_as_spelling: pron 비어있음 = 철자=발음 = 삽입 없음
    - internal_unknown: 단어 내부 후보 (형태소 경계 없어 판단 불가)
    """
    candidates = []
    parse_fail_count = 0

    # 명사만 필터링
    df_noun = df[df['pos'] == '명사'].copy()
    print(f"명사: {len(df_noun):,}개")

    # dict_morph가 있고 + 또는 -를 포함하는 것만 (복합명사)
    df_compound = df_noun[df_noun['dict_morph'].notna()]
    df_compound = df_compound[df_compound['dict_morph'].str.contains(r'[-+]', na=False)]
    print(f"dict_morph에 형태소 경계가 있는 명사: {len(df_compound):,}개")

    # word_roman이 있는 것만
    df_compound = df_compound[df_compound['word_roman'].notna()]
    print(f"word_roman도 있는 명사: {len(df_compound):,}개")

    # === Pass 1: 형태소 경계 기반 검색 ===
    print("\n[Pass 1] 형태소 경계 기반 검색...")
    for idx, row in df_compound.iterrows():
        dict_morph = row['dict_morph']
        word_roman = row['word_roman']

        # 경계 유형 정보 포함
        morphemes_kor, boundaries = parse_dict_morph_with_boundaries(dict_morph)

        # FIXED: 형태소 단위로 재구성 (음절 vs 형태소 문제 해결)
        morphemes_roman = parse_word_roman_by_morphemes(word_roman, morphemes_kor)

        if len(morphemes_roman) < 2:
            if len(morphemes_kor) >= 2:
                parse_fail_count += 1
            continue

        env_list = check_n_l_insertion_env(morphemes_kor, morphemes_roman)

        if env_list:
            for m1_kor, m2_kor, m1_roman, m2_roman, ins_type, pos in env_list:
                # 해당 위치의 경계 유형 가져오기
                boundary_type = boundaries[pos] if pos < len(boundaries) else ''

                # 사전 발음 정보
                pron_raw = row.get('pron', '')
                pron = str(pron_raw) if pd.notna(pron_raw) else ''
                pron_roman = str(row.get('pron_roman', '')) if pd.notna(row.get('pron_roman', '')) else ''

                # 발음 로마자로 실제 삽입 여부 자동 감지
                detected_insertion = detect_insertion_from_pron(m1_roman, m2_roman, pron_roman, boundary_type='morpheme')

                # 기존: 단순히 'ㄴ' 글자가 있는지만 체크 (참고용)
                has_n_in_pron = 'ㄴ' in pron
                has_l_in_pron = 'ㄹ' in pron

                # 어종 분류: sense_origin_dict 기반
                compound_type, morph1_origin, morph2_origin = classify_compound_type_from_row(row, sense_origin_dict)

                # 조건 변수 컬럼
                m2_onset_type = get_m2_onset_type(m2_roman)
                m1_coda_type = get_m1_coda_type(m1_roman)
                m2_vowel_height = get_m2_vowel_height(m2_roman)
                branching = get_branching_structure(morphemes_kor, boundaries) if len(morphemes_kor) >= 3 else ''
                m1_syllable_count = len(m1_roman.split('-'))
                m2_syllable_count = len(m2_roman.split('-'))

                candidates.append({
                    # 기본 정보
                    'word': row['word'],
                    'word_stem': row['word_stem'],
                    'pron': pron,
                    'pron_roman': pron_roman,
                    'insertion_type': ins_type,

                    # 자동 감지 결과 (로마자 기반)
                    'detected_insertion': detected_insertion,

                    # 기존 (참고용, 부정확할 수 있음)
                    'n_in_pron': 'yes' if has_n_in_pron else 'no',
                    'l_in_pron': 'yes' if has_l_in_pron else 'no',

                    'sense_id': row.get('sense_id', ''),
                    'definition': row.get('definition', ''),

                    # 형태소 분석 (비교용)
                    'dict_morph': dict_morph,
                    'word_roman': word_roman,
                    'seg_morph': row.get('seg_morph', ''),
                    'anal_morph': row.get('anal_morph', ''),
                    'seg_status': row.get('seg_status', ''),
                    'seg_links': row.get('seg_links', ''),

                    # 파싱된 형태소
                    'morph1': m1_kor,
                    'morph2': m2_kor,
                    'morph1_roman': m1_roman,
                    'morph2_roman': m2_roman,
                    'position': pos,

                    # 형태론 (어종 정보)
                    'morph1_origin': morph1_origin,
                    'morph2_origin': morph2_origin,
                    'compound_type': compound_type,

                    # 경계 유형 (합성/파생)
                    'boundary_type': boundary_type,

                    # 조건 변수
                    'm2_onset_type': m2_onset_type,
                    'm1_coda_type': m1_coda_type,
                    'm2_vowel_height': m2_vowel_height,
                    'branching': branching,
                    'm1_syllable_count': m1_syllable_count,
                    'm2_syllable_count': m2_syllable_count,

                    # LS 빈도
                    'freq_LS_messenger': row.get('freq_MXLS', 0),
                    'freq_LS_written': row.get('freq_NXLS', 0),
                    'freq_LS_spoken': row.get('freq_SXLS', 0),
                    'freq_LS_total': row.get('freq_LS_total', 0),

                    # MP 빈도
                    'freq_MP_written': row.get('freq_MP_NXMP', 0),
                    'freq_MP_spoken': row.get('freq_MP_SXMP', 0),
                    'freq_MP_total': row.get('freq_MP_total', 0),

                    # Freq_2009
                    'freq_06b': row.get('freq_06b', 0),
                    'freq_13a': row.get('freq_13a', 0),
                })

    print(f"  형태소 경계 후보: {len(candidates):,}개")
    if parse_fail_count > 0:
        print(f"  [주의] word_roman 파싱 실패로 제외된 복합어: {parse_fail_count:,}개")

    # === Pass 2: 단어 내부 ㄴ삽입 검색 ===
    print("\n[Pass 2] 단어 내부 ㄴ삽입 검색...")
    df_noun_roman = df_noun[df_noun['word_roman'].notna()]
    internal_count = 0

    for idx, row in df_noun_roman.iterrows():
        word_roman = row['word_roman']
        internal_results = check_n_l_insertion_internal(word_roman)

        if internal_results:
            for result in internal_results:
                pron_raw = row.get('pron', '')
                pron = str(pron_raw) if pd.notna(pron_raw) else ''
                pron_roman = str(row.get('pron_roman', '')) if pd.notna(row.get('pron_roman', '')) else ''

                # left_roman/right_roman이 이제 check_n_l_insertion_internal에서 반환됨
                left_roman = result.get('left_roman', '')
                right_roman = result.get('right_roman', '')

                detected = detect_insertion_from_pron(
                    left_roman,
                    right_roman,
                    pron_roman,
                    boundary_type='internal'
                )

                # 내부 후보도 word_type 기반 어종 추정
                wt = row.get('word_type', '')
                internal_origin = _extract_origin_from_word_type(str(wt)) if pd.notna(wt) and wt else 'unknown'

                candidates.append({
                    # 기본 정보
                    'word': row['word'],
                    'word_stem': row['word_stem'],
                    'pron': pron,
                    'pron_roman': pron_roman,
                    'insertion_type': result.get('insertion_type', 'ㄴ삽입'),

                    # 자동 감지
                    'detected_insertion': detected,

                    # 참고용
                    'n_in_pron': 'yes' if 'ㄴ' in pron else 'no',
                    'l_in_pron': 'yes' if 'ㄹ' in pron else 'no',

                    'sense_id': row.get('sense_id', ''),
                    'definition': row.get('definition', ''),

                    # 형태소 분석
                    'dict_morph': row.get('dict_morph', ''),
                    'word_roman': word_roman,
                    'seg_morph': row.get('seg_morph', ''),
                    'anal_morph': row.get('anal_morph', ''),
                    'seg_status': row.get('seg_status', ''),
                    'seg_links': row.get('seg_links', ''),

                    # 파싱된 형태소 (내부 경계)
                    'morph1': result.get('syllable1', ''),
                    'morph2': result.get('syllable2', ''),
                    'morph1_roman': left_roman,
                    'morph2_roman': right_roman,
                    'position': result.get('position', -1),

                    # 형태론 (word_type 기반 추정)
                    'morph1_origin': internal_origin,
                    'morph2_origin': internal_origin,
                    'compound_type': f'{internal_origin}(internal)',

                    # 경계 유형
                    'boundary_type': 'internal',

                    # 조건 변수
                    'm2_onset_type': get_m2_onset_type(right_roman),
                    'm1_coda_type': get_m1_coda_type(left_roman),
                    'm2_vowel_height': get_m2_vowel_height(right_roman),
                    'branching': '',
                    'm1_syllable_count': len(left_roman.split('-')) if left_roman else 0,
                    'm2_syllable_count': len(right_roman.split('-')) if right_roman else 0,

                    # LS 빈도
                    'freq_LS_messenger': row.get('freq_MXLS', 0),
                    'freq_LS_written': row.get('freq_NXLS', 0),
                    'freq_LS_spoken': row.get('freq_SXLS', 0),
                    'freq_LS_total': row.get('freq_LS_total', 0),

                    # MP 빈도
                    'freq_MP_written': row.get('freq_MP_NXMP', 0),
                    'freq_MP_spoken': row.get('freq_MP_SXMP', 0),
                    'freq_MP_total': row.get('freq_MP_total', 0),

                    # Freq_2009
                    'freq_06b': row.get('freq_06b', 0),
                    'freq_13a': row.get('freq_13a', 0),
                })
                internal_count += 1

    print(f"  단어 내부 후보: {internal_count:,}개")

    return pd.DataFrame(candidates)

print("\n검색 함수 준비 완료 (sense_origin_dict 기반 + 조건 변수 + 단어 내부 ㄴ삽입)")

In [55]:
# 4.3 검색 실행
print("ㄴ/ㄹ 삽입 후보 검색 중...\n")
df_candidates = search_n_l_insertion_candidates(df_v7, sense_origin_dict)

print(f"\n✅ 검색 완료: {len(df_candidates):,}개 후보")
print(f"  ㄴ삽입: {(df_candidates['insertion_type'] == 'ㄴ삽입').sum():,}개")
print(f"  ㄹ삽입: {(df_candidates['insertion_type'] == 'ㄹ삽입').sum():,}개")

print(f"\n【합성어 유형별】")
print(df_candidates['compound_type'].value_counts())

print(f"\n【경계 유형별】")
print(df_candidates['boundary_type'].value_counts())

print(f"\n【어종 분포】")
print(f"  morph1_origin:")
print(df_candidates['morph1_origin'].value_counts())
print(f"\n  morph2_origin:")
print(df_candidates['morph2_origin'].value_counts())

ㄴ/ㄹ 삽입 후보 검색 중...

명사: 395,985개
dict_morph에 형태소 경계가 있는 명사: 214,557개
word_roman도 있는 명사: 214,557개

[Pass 1] 형태소 경계 기반 검색...
  형태소 경계 후보: 6,628개

[Pass 2] 단어 내부 ㄴ삽입 검색...
  단어 내부 후보: 26,133개

✅ 검색 완료: 32,761개 후보
  ㄴ삽입: 27,437개
  ㄹ삽입: 5,324개

【합성어 유형별】
compound_type
       26133
S+S     4236
U+U     1213
N+N     1159
L+L       20
Name: count, dtype: int64

【경계 유형별】
boundary_type
internal    26133
compound     6628
Name: count, dtype: int64

【어종 분포】
  morph1_origin:
morph1_origin
           26133
한자어         4236
unknown     1213
고유어         1159
외래어           20
Name: count, dtype: int64

  morph2_origin:
morph2_origin
           26133
한자어         4236
unknown     1213
고유어         1159
외래어           20
Name: count, dtype: int64


In [56]:
# 4.4 결과 미리보기
print("\n상위 20개 (빈도순):\n")
df_candidates[['word', 'dict_morph', 'word_roman', 'insertion_type', 'detected_insertion', 'pron', 'freq_LS_total']].sort_values('freq_LS_total', ascending=False).head(20)


상위 20개 (빈도순):



,word,dict_morph,word_roman,insertion_type,detected_insertion,pron,freq_LS_total
11240,필요,NaN,PhIl-iO,ㄹ삽입,no,피료,2866
7971,중요,NaN,JUng-iO,ㄴ삽입,no,중ː요,1942
13896,금융,NaN,GEUm-iUng,ㄴ삽입,no,금늉,1651
12308,확인,NaN,HoAk-In,ㄴ삽입,no,화긴,1536
8727,참여,NaN,ChAm-iEO,ㄴ삽입,no,차며,1503
29035,운영,NaN,Un-iEOng,ㄴ삽입,no,우ː녕,1426
22066,분야,NaN,BUn-iA,ㄴ삽입,no,부냐,1092
7609,공연,NaN,GOng-iEOn,ㄴ삽입,no,공연,839
12401,활용,NaN,HoAl-iOng,ㄹ삽입,no,화룡,835
31370,적용,NaN,JEOk-iOng,ㄴ삽입,no,저굥,834


---

## 5️⃣ 형태소 빈도 추가

In [57]:
# 5.1 형태소 빈도 매칭
def add_morpheme_freq(df, morph_freq_dict):
    df = df.copy()
    df['freq_morph1_LS'] = df['morph1'].map(morph_freq_dict).fillna(0).astype(int)
    df['freq_morph2_LS'] = df['morph2'].map(morph_freq_dict).fillna(0).astype(int)
    return df

df_candidates = add_morpheme_freq(df_candidates, morph_freq_agg)

print("✅ 형태소 빈도 추가 완료")
print(f"morph1 빈도 있는 행: {(df_candidates['freq_morph1_LS'] > 0).sum():,}개")
print(f"morph2 빈도 있는 행: {(df_candidates['freq_morph2_LS'] > 0).sum():,}개")

✅ 형태소 빈도 추가 완료
morph1 빈도 있는 행: 4,402개
morph2 빈도 있는 행: 5,779개


---

## 6️⃣ 결과 저장

In [58]:
# 6.1 CSV 저장
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'{RESULT_DIR}/n_l_insertion_candidates_v2_{timestamp}.csv'

# 검토용 빈 컬럼 추가
df_candidates['review_status'] = 'pending'
df_candidates['morph_ok'] = ''
df_candidates['n_insertion'] = ''  # 수동 검토용 (최종 판단)
df_candidates['decision'] = ''
df_candidates['notes'] = ''

# 컬럼 순서 정리
column_order = [
    # 기본
    'word', 'word_stem', 'sense_id',

    # 발음 및 유형
    'insertion_type', 'pron', 'pron_roman',
    'detected_insertion',  # 자동 감지 (로마자 기반)
    'n_in_pron', 'l_in_pron',  # 기존 (참고용)

    # 형태소 분석 (비교용)
    'dict_morph', 'word_roman',
    'morph1', 'morph2', 'morph1_roman', 'morph2_roman', 'position',

    # 형태론 (어종 정보)
    'morph1_origin', 'morph2_origin', 'compound_type',

    # 경계 유형 (합성/파생)
    'boundary_type',

    # 조건 변수
    'm2_onset_type', 'm1_coda_type', 'm2_vowel_height',
    'branching', 'm1_syllable_count', 'm2_syllable_count',

    'seg_morph', 'anal_morph', 'seg_status', 'seg_links',

    # 빈도
    'freq_LS_messenger', 'freq_LS_written', 'freq_LS_spoken', 'freq_LS_total',
    'freq_MP_written', 'freq_MP_spoken', 'freq_MP_total',
    'freq_06b', 'freq_13a',  # Freq_2009 (강범모·김흥규 2009)
    'freq_morph1_LS', 'freq_morph2_LS',

    # 뜻풀이
    'definition',

    # 수동 검토용
    'review_status', 'morph_ok', 'n_insertion', 'decision', 'notes'
]

df_full = df_candidates[column_order].sort_values('freq_LS_total', ascending=False)
df_full.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 결과 저장: {output_path}")
print(f"   총 {len(df_full):,}개 후보")
print(f"\n📋 개선 사항:")
print(f"   ✅ utils_phonology.py 공유 모듈 활용")
print(f"   ✅ sense_origin_dict 기반 어종 분류 (동음이의어 해소)")
print(f"   ✅ 조건 변수 추가: m2_onset_type, m1_coda_type, m2_vowel_height")
print(f"   ✅ branching 구조 (3형태소 이상)")
print(f"   ✅ m1/m2 음절 수")
print(f"   ✅ 단어 내부 ㄴ삽입 검색 (boundary_type='internal')")
print(f"\n💡 boundary_type 값:")
print(f"   - compound: 합성어 경계 (-)")
print(f"   - derivation: 파생어 경계 (+)")
print(f"   - internal: 단어 내부 (형태소 경계 없이 음절간)")
print(f"\n⚠️  수동 검토 가이드: REVIEW_GUIDE_V2.md 참고")

✅ 결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/n_l_insertion_candidates_v2_20260312_070223.csv
   총 32,761개 후보

📋 개선 사항:
   ✅ utils_phonology.py 공유 모듈 활용
   ✅ sense_origin_dict 기반 어종 분류 (동음이의어 해소)
   ✅ 조건 변수 추가: m2_onset_type, m1_coda_type, m2_vowel_height
   ✅ branching 구조 (3형태소 이상)
   ✅ m1/m2 음절 수
   ✅ 단어 내부 ㄴ삽입 검색 (boundary_type='internal')

💡 boundary_type 값:
   - compound: 합성어 경계 (-)
   - derivation: 파생어 경계 (+)
   - internal: 단어 내부 (형태소 경계 없이 음절간)

⚠️  수동 검토 가이드: REVIEW_GUIDE_V2.md 참고


In [59]:
# 6.2 통계 요약
print("\n" + "="*70)
print("ㄴ/ㄹ 삽입 후보 검색 요약 (v2)")
print("="*70)
print(f"총 후보 수: {len(df_full):,}개")

print(f"\n【삽입 유형 (환경 기준)】")
print(df_full['insertion_type'].value_counts())

print(f"\n【자동 감지 결과 (pron_roman 기반)】")
print(df_full['detected_insertion'].value_counts())
print()
print("  - n_inserted: 사전 발음에서 ㄴ 삽입 확인")
print("  - l_inserted: 사전 발음에서 ㄹ 삽입 확인")
print("  - no: pron 있지만 삽입 아닌 다른 현상")
print("  - no_pron_same_as_spelling: pron 비어있음 = 철자=발음 = 삽입 없음")
print("  - internal_unknown: 단어 내부 후보 (형태소 경계 없어 판단 불가)")

print(f"\n【합성명사 빈도 분포】")
print(f"  LS_total > 0: {(df_full['freq_LS_total'] > 0).sum():,}개")
print(f"  LS_total >= 10: {(df_full['freq_LS_total'] >= 10).sum():,}개")
print(f"  LS_total >= 100: {(df_full['freq_LS_total'] >= 100).sum():,}개")

print(f"\n【형태소 경계 후보 감지 현황】")
df_morph = df_full[df_full['boundary_type'] != 'internal']
if len(df_morph) > 0:
    print(f"  형태소 경계 후보: {len(df_morph):,}개")
    print(df_morph['detected_insertion'].value_counts().to_string())

print(f"\n【단어 내부 후보 감지 현황】")
df_internal = df_full[df_full['boundary_type'] == 'internal']
if len(df_internal) > 0:
    print(f"  단어 내부 후보: {len(df_internal):,}개")
    print(df_internal['detected_insertion'].value_counts().to_string())

print("="*70)


ㄴ/ㄹ 삽입 후보 검색 요약 (v2)
총 후보 수: 32,761개

【삽입 유형 (환경 기준)】
insertion_type
ㄴ삽입    27437
ㄹ삽입     5324
Name: count, dtype: int64

【자동 감지 결과 (pron_roman 기반)】
detected_insertion
no                          25842
n_inserted                   3560
internal_unknown             1946
l_inserted                    918
no_pron_same_as_spelling      495
Name: count, dtype: int64

  - n_inserted: 사전 발음에서 ㄴ 삽입 확인
  - l_inserted: 사전 발음에서 ㄹ 삽입 확인
  - no: pron 있지만 삽입 아닌 다른 현상
  - no_pron_same_as_spelling: pron 비어있음 = 철자=발음 = 삽입 없음
  - internal_unknown: 단어 내부 후보 (형태소 경계 없어 판단 불가)

【합성명사 빈도 분포】
  LS_total > 0: 3,136개
  LS_total >= 10: 781개
  LS_total >= 100: 155개

【형태소 경계 후보 감지 현황】
  형태소 경계 후보: 6,628개
detected_insertion
n_inserted                  3560
no                          1655
l_inserted                   918
no_pron_same_as_spelling     495

【단어 내부 후보 감지 현황】
  단어 내부 후보: 26,133개
detected_insertion
no                  24187
internal_unknown     1946
